# Closing the Loop: Turning Production Failures Into Automated Prompt Improvements

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/falcon-ai-page/optimization/closing-the-loop-prod-failures.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/falcon-ai-page/optimization/closing-the-loop-prod-failures.ipynb)

| Time | Difficulty |
|------|------------|
| 20 min | Intermediate |

By the end of this cookbook you will have an automated loop that takes a set of failing production traces, runs `agent-opt` to find a better prompt that fixes them (scored by the same eval that flagged them), and verifies the lift on a held-out batch. Same pipeline runs in CI on every release so a regression in production becomes a fix proposal in the next deploy.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY`
- A traced Observe project with at least one continuous eval task running
- An OpenAI API key
- Python 3.9+

## Install

Install the optimization SDK, eval SDK, and `requests`.

In [ ]:
%pip install agent-opt ai-evaluation requests

In [ ]:
import os
os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["OPENAI_API_KEY"] = "your-openai-key"


## Step 1: Define the regression dataset

The input to the loop is a list of user messages where your agent failed. In a real loop you'd pull these from your Observe project (dashboard export to CSV, or the MCP server's `search_traces` tool when running this in CI), but to keep this cookbook self-contained we'll hardcode 15 specific, answerable support questions paired with the **deflecting** replies prod logged. These are the kind of "I don't know" / "Probably" / "Check our policy" responses that `is_helpful` correctly flags as Failed.

When you have real failures, swap `RAW_FAILURES` below for the rows from your dashboard export. The rest of the cookbook stays the same.

In [ ]:
API_KEY = os.environ["FI_API_KEY"]
SECRET_KEY = os.environ["FI_SECRET_KEY"]
EVAL_NAME = "is_helpful"

RAW_FAILURES = [
    {"input": "What is your return window?",                      "output": "I don't know."},
    {"input": "How long does standard shipping take?",            "output": "Could be a while."},
    {"input": "Do you accept Apple Pay?",                         "output": "Maybe."},
    {"input": "What's the difference between Pro and Enterprise?","output": "They're different plans."},
    {"input": "How do I reset my password?",                      "output": "Search the help center."},
    {"input": "Is my account data encrypted?",                    "output": "Probably."},
    {"input": "Can I get a refund 30 days after purchase?",       "output": "Check our policy."},
    {"input": "Do you ship to Canada?",                           "output": "Not sure."},
    {"input": "How do I cancel my subscription?",                 "output": "Look in your account settings."},
    {"input": "Is there a free trial?",                           "output": "Maybe, I think so."},
    {"input": "What payment methods do you accept?",              "output": "Various ones."},
    {"input": "How do I update my billing address?",              "output": "Probably in settings."},
    {"input": "Do you have a mobile app?",                        "output": "Possibly."},
    {"input": "What's the storage limit on the basic plan?",      "output": "Some amount."},
    {"input": "Can I export my data as CSV?",                     "output": "I'd have to check."},
]

print(f"Loaded {len(RAW_FAILURES)} failing traces")
for row in RAW_FAILURES[:3]:
    print(f"  input: {row['input']!r:45}  broken output: {row['output']!r}")


## Step 2: Split into train + held-out

The optimizer can overfit to whatever rows it sees, so split the failures into a training set (rows the optimizer scores against) and a held-out set (rows we'll use in step 4 to verify the lift is real). Two non-overlapping batches let us measure generalization, not just memorization.

In [ ]:
rows = [
    {"user_input": r["input"], "agent_output": r["output"]}
    for r in RAW_FAILURES
]

HELD_OUT_SIZE = max(len(rows) // 3, 1)
dataset = rows[:-HELD_OUT_SIZE]
held_out = rows[-HELD_OUT_SIZE:]

print(f"Train dataset:  {len(dataset)} rows")
print(f"Held-out set:   {len(held_out)} rows")
print(f"\nFirst train row: {dataset[0]}")


## Step 3: Run prompt optimization with the eval as the objective

`agent-opt` takes four things: a generator (model that runs the candidate prompt), an evaluator (scores its output), a data mapper (matches dataset keys to eval inputs), and an optimizer (search strategy). The eval template you pick must be the SAME one whose failures you pulled in step 1, otherwise the optimizer is solving a different problem than your alert monitor cares about.

In [ ]:
from fi.opt.base.evaluator import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import MetaPromptOptimizer

# 1. The same eval that flagged the failures becomes the optimization objective.
evaluator = Evaluator(
    eval_template=EVAL_NAME,
    eval_model_name="turing_flash",
    fi_api_key=API_KEY,
    fi_secret_key=SECRET_KEY,
)

# 2. Map dataset keys to what the eval template expects.
data_mapper = BasicDataMapper(
    key_map={"input": "user_input", "output": "generated_output"},
)

# 3. The generator runs candidate prompts and produces outputs to score.
initial_prompt = "You are a customer support agent. Answer the user's question: {user_input}"
generator = LiteLLMGenerator(model="gpt-4o-mini", prompt_template=initial_prompt)

# 4. Meta-Prompt: a teacher model reads failing scores and rewrites the prompt each round.
teacher = LiteLLMGenerator(model="gpt-4o", prompt_template="{prompt}")
optimizer = MetaPromptOptimizer(teacher_generator=teacher)

# Cost per round = eval_subset_size × (1 generator call + 1 judge call) + 1 teacher call.
# Defaults are 5 rounds × 40-row subsets; scale these to your dataset and budget.
result = optimizer.optimize(
    evaluator=evaluator,
    data_mapper=data_mapper,
    dataset=dataset,
    initial_prompts=[initial_prompt],
    task_description="Customer support replies that are on-topic and resolve the user's question.",
    num_rounds=2,
    eval_subset_size=5,
)

print(f"Initial score:  {result.history[0].average_score:.3f}")
print(f"Final score:    {result.final_score:.3f}")
print(f"\nBest prompt:\n{result.best_generator.get_prompt_template()}")


## Step 4: Verify the new prompt actually fixes the regressions

Optimizers can overfit to the small dataset they trained on. The held-out batch we set aside in step 2 is rows the optimizer never saw. The right comparison is "what the broken prompt actually produced in prod" (the `agent_output` we logged) vs "what the new prompt produces now" — score both with the same eval. The broken outputs were by definition flagged Failed, so any pass rate on the new prompt is real uplift.

In [ ]:
from fi.evals import Evaluator as RuntimeEvaluator
from openai import OpenAI

runtime_eval = RuntimeEvaluator(fi_api_key=API_KEY, fi_secret_key=SECRET_KEY)
client = OpenAI()


def is_passed(user_input, agent_output):
    r = runtime_eval.evaluate(
        eval_templates=EVAL_NAME,
        inputs={"input": user_input, "output": agent_output},
        model_name="turing_flash",
    )
    return str(r.eval_results[0].output).strip().lower() == "passed"


def generate(prompt_template, user_input):
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_template.format(user_input=user_input)}],
    ).choices[0].message.content


new_prompt = result.best_generator.get_prompt_template()

# Baseline: the deflecting replies prod actually logged for these inputs.
old_pass = sum(1 for row in held_out if is_passed(row["user_input"], row["agent_output"]))
# Treatment: fresh outputs from the optimized prompt on the same inputs.
new_pass = sum(1 for row in held_out if is_passed(row["user_input"], generate(new_prompt, row["user_input"])))

n = max(len(held_out), 1)
print(f"Held-out batch: {len(held_out)} rows")
print(f"Logged broken outputs pass rate: {old_pass}/{len(held_out)} ({100*old_pass/n:.0f}%)")
print(f"New prompt pass rate:            {new_pass}/{len(held_out)} ({100*new_pass/n:.0f}%)")


> **Check.** Failing traces from production pulled programmatically, fed to `agent-opt` as a regression dataset, optimized against the same eval that flagged them, and verified on a held-out batch with measurable score uplift before shipping. End-to-end loop in one script.

Ship the new prompt when the held-out improvement is large enough to be statistically meaningful (a 30-row batch needs roughly 6+ percentage points of uplift to clear noise). If the lift is smaller, the optimizer probably overfit. Pull more failing traces and re-run.

## Explore further

- **[Optimization SDK](https://docs.futureagi.com/docs/optimization/features/using-python-sdk)**: `agent-opt` reference: optimizers, generators, data mappers
- **[Eval Correction Loop](https://docs.futureagi.com/docs/cookbook/evaluation/eval-correction-loop)**: Calibrate the eval first if generic templates miss your domain rules
- **[Continuous Evals on a Budget](https://docs.futureagi.com/docs/cookbook/observe/continuous-evals-budget)**: Set up the production eval task that produces the failing traces